## Objective

This notebook translates the predictive, cross-country, domain-shift, and explainability findings into a structured road-safety intervention framework.

The framework follows the sequence:

**Observed Risk Pattern → Predictive Evidence → SHAP Evidence → Modifiability → Intervention Priority → Evidence-Based Intervention**

The analysis distinguishes between:

1. Road and infrastructure factors
2. Vehicle and collision-interaction factors
3. Temporal and environmental factors
4. Driver-related factors

Intervention recommendations are not inferred directly from SHAP values. SHAP is used to identify model-relevant factors, while intervention selection requires consideration of modifiability, severity relevance, stability, cross-country evidence, and external road-safety evidence.

Driver-related findings from Ethiopia are treated separately because these variables were unavailable in the harmonized UK–France–Ethiopia feature space.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
PROJECT_ROOT = Path("..")

TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Table directory:")
print(TABLE_DIR.resolve())

print("\nFigure directory:")
print(FIGURE_DIR.resolve())

In [ ]:
# List previously generated analytical tables

csv_files = sorted(TABLE_DIR.glob("*.csv"))

print(f"Number of saved CSV tables: {len(csv_files)}")
print("=" * 80)

for i, file in enumerate(csv_files, start=1):
    print(f"{i:02d}. {file.name}")

In [ ]:
# Cell 4 — Load core evidence tables

import pandas as pd

# ============================================================
# UK SHAP
# ============================================================

uk_global_shap = pd.read_csv(
    TABLE_DIR / "Table_UK_Global_SHAP_Importance.csv"
)

uk_class_shap = pd.read_csv(
    TABLE_DIR / "Table_UK_Class_Specific_SHAP_Comparison.csv"
)

uk_shap_stability = pd.read_csv(
    TABLE_DIR / "Table_UK_SHAP_Stability_Summary.csv"
)


# ============================================================
# Ethiopia SHAP
# ============================================================

eth_global_shap = pd.read_csv(
    TABLE_DIR / "Table_Ethiopia_Global_SHAP_Importance.csv"
)

eth_class_shap = pd.read_csv(
    TABLE_DIR / "Table_Ethiopia_Class_Specific_SHAP.csv"
)

eth_shap_stability = pd.read_csv(
    TABLE_DIR / "Table_Ethiopia_SHAP_Stability_Summary.csv"
)

eth_group_stability = pd.read_csv(
    TABLE_DIR / "Table_Ethiopia_SHAP_Group_Stability_Summary.csv"
)

eth_driver_ablation = pd.read_csv(
    TABLE_DIR / "Table_Ethiopia_Driver_Ablation.csv"
)

eth_abc_cv = pd.read_csv(
    TABLE_DIR / "Table_Ethiopia_ABC_CV_Summary.csv"
)


# ============================================================
# Cross-country evidence
# ============================================================

domain_shift = pd.read_csv(
    TABLE_DIR / "Table_Three_Country_Feature_Domain_Shift.csv"
)

overall_shift = pd.read_csv(
    TABLE_DIR / "Table_Three_Country_Overall_Domain_Shift.csv"
)

transfer_results = pd.read_csv(
    TABLE_DIR / "Table_Six_Direction_Cross_Country_Transfer.csv"
)

severity_summary = pd.read_csv(
    TABLE_DIR / "cross_country_dataset_summary.csv"
)


# ============================================================
# Collision configuration
# ============================================================

uk_collision = pd.read_csv(
    TABLE_DIR / "Table_UK_Collision_Configuration_Severity.csv"
)

france_collision = pd.read_csv(
    TABLE_DIR / "Table_France_Collision_Configuration_Severity.csv"
)


print("Core evidence tables loaded successfully.")

In [ ]:
# Cell 5 — Inspect actual table schemas

tables_to_check = {
    "UK Global SHAP": uk_global_shap,
    "UK Class SHAP": uk_class_shap,
    "UK SHAP Stability": uk_shap_stability,

    "Ethiopia Global SHAP": eth_global_shap,
    "Ethiopia Class SHAP": eth_class_shap,
    "Ethiopia SHAP Stability": eth_shap_stability,
    "Ethiopia Driver Ablation": eth_driver_ablation,
    "Ethiopia ABC CV": eth_abc_cv,

    "Domain Shift": domain_shift,
    "Overall Domain Shift": overall_shift,
    "Transfer Results": transfer_results,

    "UK Collision": uk_collision,
    "France Collision": france_collision
}

for name, df in tables_to_check.items():

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 3 rows:")
    print(df.head(3).to_string(index=False))

In [ ]:
# Cell 6 — Build a unified evidence master table from saved UK, Ethiopia,
# domain-shift, and transferability results.

# ---------------------------------------------------------
# UK evidence
# ---------------------------------------------------------

uk_master = (
    uk_global_shap[
        ["Feature", "Importance_Percent", "Rank"]
    ]
    .rename(columns={
        "Importance_Percent": "UK_Global_SHAP_%",
        "Rank": "UK_Global_Rank"
    })
)

uk_master = uk_master.merge(
    uk_class_shap[
        [
            "Feature",
            "Fatal_Importance_%",
            "Fatal_Rank",
            "Serious_Importance_%",
            "Serious_Rank"
        ]
    ],
    on="Feature",
    how="left"
)

uk_master = uk_master.merge(
    uk_shap_stability[
        [
            "Feature",
            "Mean_Importance_Percent",
            "SD_Importance_Percent",
            "Mean_Rank"
        ]
    ].rename(columns={
        "Mean_Importance_Percent": "UK_Stable_SHAP_%",
        "SD_Importance_Percent": "UK_SHAP_SD",
        "Mean_Rank": "UK_Stable_Rank"
    }),
    on="Feature",
    how="left"
)

uk_master["Evidence_Context"] = "UK"


# ---------------------------------------------------------
# Ethiopia evidence
# ---------------------------------------------------------

eth_fatal = (
    eth_class_shap[
        eth_class_shap["Class"] == "Fatal"
    ][
        [
            "Feature",
            "Importance_Percent",
            "Rank"
        ]
    ]
    .rename(columns={
        "Importance_Percent": "Ethiopia_Fatal_SHAP_%",
        "Rank": "Ethiopia_Fatal_Rank"
    })
)

eth_serious = (
    eth_class_shap[
        eth_class_shap["Class"] == "Serious"
    ][
        [
            "Feature",
            "Importance_Percent",
            "Rank"
        ]
    ]
    .rename(columns={
        "Importance_Percent": "Ethiopia_Serious_SHAP_%",
        "Rank": "Ethiopia_Serious_Rank"
    })
)

eth_master = (
    eth_global_shap[
        [
            "Feature",
            "Feature_Group",
            "Importance_Percent",
            "Rank"
        ]
    ]
    .rename(columns={
        "Importance_Percent": "Ethiopia_Global_SHAP_%",
        "Rank": "Ethiopia_Global_Rank"
    })
)

eth_master = eth_master.merge(
    eth_fatal,
    on="Feature",
    how="left"
)

eth_master = eth_master.merge(
    eth_serious,
    on="Feature",
    how="left"
)

eth_master = eth_master.merge(
    eth_shap_stability[
        [
            "Feature",
            "Mean_Importance_Percent",
            "SD_Importance_Percent",
            "Mean_Rank"
        ]
    ].rename(columns={
        "Mean_Importance_Percent": "Ethiopia_Stable_SHAP_%",
        "SD_Importance_Percent": "Ethiopia_SHAP_SD",
        "Mean_Rank": "Ethiopia_Stable_Rank"
    }),
    on="Feature",
    how="left"
)

eth_master["Evidence_Context"] = "Ethiopia"


# ---------------------------------------------------------
# Domain-shift evidence for harmonized features
# ---------------------------------------------------------

domain_master = domain_shift[
    [
        "Feature",
        "UK vs France",
        "UK vs Ethiopia",
        "France vs Ethiopia",
        "Mean_JS_Distance"
    ]
].copy()

domain_master = domain_master.rename(columns={
    "UK vs France": "JS_UK_France",
    "UK vs Ethiopia": "JS_UK_Ethiopia",
    "France vs Ethiopia": "JS_France_Ethiopia",
    "Mean_JS_Distance": "Mean_JS"
})


# ---------------------------------------------------------
# Combine UK and Ethiopia evidence
# ---------------------------------------------------------

evidence_master = pd.merge(
    uk_master,
    eth_master,
    on="Feature",
    how="outer",
    suffixes=("_UK", "_ETH")
)

evidence_master = evidence_master.merge(
    domain_master,
    on="Feature",
    how="left"
)


# ---------------------------------------------------------
# Add high-level factor domains
# ---------------------------------------------------------

factor_domain_map = {
    "junction_common": "Road/Infrastructure",
    "surface_common": "Road/Infrastructure",
    "Road_surface_type": "Road/Infrastructure",
    "Road_allignment": "Road/Infrastructure",
    "Area_accident_occured": "Road/Infrastructure",

    "vehicle_count": "Vehicle/Collision",
    "vehicle_category": "Vehicle/Collision",
    "has_motorcycle": "Vehicle/Collision",
    "has_two_wheeler": "Vehicle/Collision",
    "has_heavy_vehicle": "Vehicle/Collision",
    "has_public_transport": "Vehicle/Collision",
    "Type_of_collision": "Vehicle/Collision",
    "Vehicle_movement": "Vehicle/Collision",
    "Number_of_casualties": "Crash Consequence",

    "weather_common": "Temporal/Environmental",
    "light_common": "Temporal/Environmental",
    "hour": "Temporal/Environmental",
    "day_common": "Temporal/Environmental",

    "Age_band_of_driver": "Driver",
    "Driving_experience": "Driver",
    "Educational_level": "Driver",
    "Drivers_gender": "Driver",
    "Vehicle_driver_relation": "Driver",

    "Cause_of_accident": "Crash/Behavioral"
}

evidence_master["Factor_Domain"] = (
    evidence_master["Feature"]
    .map(factor_domain_map)
    .fillna("Other")
)


# ---------------------------------------------------------
# Modifiability classification
# ---------------------------------------------------------

modifiability_map = {
    "junction_common": "Directly modifiable",
    "surface_common": "Directly modifiable",
    "Road_surface_type": "Directly modifiable",
    "Road_allignment": "Directly modifiable",
    "Area_accident_occured": "Partly modifiable",

    "vehicle_count": "Indirectly modifiable",
    "vehicle_category": "Indirectly modifiable",
    "has_motorcycle": "Indirectly modifiable",
    "has_two_wheeler": "Indirectly modifiable",
    "has_heavy_vehicle": "Indirectly modifiable",
    "has_public_transport": "Indirectly modifiable",
    "Type_of_collision": "Indirectly modifiable",
    "Vehicle_movement": "Indirectly modifiable",

    "weather_common": "Not modifiable — targetable",
    "light_common": "Directly/partly modifiable",
    "hour": "Not modifiable — targetable",
    "day_common": "Not modifiable — targetable",

    "Age_band_of_driver": "Not modifiable — targetable",
    "Driving_experience": "Partly modifiable",
    "Educational_level": "Indirectly modifiable",
    "Drivers_gender": "Not modifiable — targetable",
    "Vehicle_driver_relation": "Indirectly modifiable",

    "Cause_of_accident": "Potentially modifiable",
    "Number_of_casualties": "Outcome-related"
}

evidence_master["Modifiability"] = (
    evidence_master["Feature"]
    .map(modifiability_map)
    .fillna("To be classified")
)


# ---------------------------------------------------------
# Sort using the strongest available Fatal SHAP evidence
# ---------------------------------------------------------

evidence_master["Fatal_Relevance"] = (
    evidence_master[
        [
            "Fatal_Importance_%",
            "Ethiopia_Fatal_SHAP_%"
        ]
    ]
    .max(axis=1, skipna=True)
)

evidence_master = (
    evidence_master
    .sort_values(
        "Fatal_Relevance",
        ascending=False,
        na_position="last"
    )
    .reset_index(drop=True)
)


print("Evidence Master Table")
print("=" * 140)

display_columns = [
    "Feature",
    "Factor_Domain",
    "Modifiability",
    "UK_Global_SHAP_%",
    "Fatal_Importance_%",
    "UK_SHAP_SD",
    "Ethiopia_Global_SHAP_%",
    "Ethiopia_Fatal_SHAP_%",
    "Ethiopia_SHAP_SD",
    "Mean_JS",
    "Fatal_Relevance"
]

print(
    evidence_master[
        display_columns
    ].round(3).to_string(index=False)
)

In [ ]:
# Cell 7 — Calculate a data-driven intervention priority score.

priority_df = evidence_master.copy()

# ---------------------------------------------------------
# 1. Severity relevance score
# Normalize Fatal relevance to 0–1
# ---------------------------------------------------------

max_fatal = priority_df["Fatal_Relevance"].max()

priority_df["Severity_Score"] = (
    priority_df["Fatal_Relevance"] / max_fatal
)


# ---------------------------------------------------------
# 2. Stability score
# Smaller SHAP SD = more stable evidence
# Use whichever country-specific stability value exists
# ---------------------------------------------------------

priority_df["Best_SHAP_SD"] = priority_df[
    ["UK_SHAP_SD", "Ethiopia_SHAP_SD"]
].min(axis=1, skipna=True)

max_sd = priority_df["Best_SHAP_SD"].max()

priority_df["Stability_Score"] = (
    1 - (priority_df["Best_SHAP_SD"] / max_sd)
)

priority_df["Stability_Score"] = (
    priority_df["Stability_Score"]
    .clip(lower=0, upper=1)
)

# Missing stability evidence = neutral rather than zero
priority_df["Stability_Score"] = (
    priority_df["Stability_Score"]
    .fillna(0.5)
)


# ---------------------------------------------------------
# 3. Actionability score
# ---------------------------------------------------------

actionability_scores = {
    "Directly modifiable": 1.00,
    "Directly/partly modifiable": 0.90,
    "Potentially modifiable": 0.85,
    "Partly modifiable": 0.75,
    "Indirectly modifiable": 0.60,
    "Not modifiable — targetable": 0.45,
    "Outcome-related": 0.00,
    "To be classified": 0.30
}

priority_df["Actionability_Score"] = (
    priority_df["Modifiability"]
    .map(actionability_scores)
    .fillna(0.30)
)


# ---------------------------------------------------------
# 4. Cross-country evidence score
# Harmonized features with JS evidence receive stronger
# cross-country relevance.
# ---------------------------------------------------------

priority_df["CrossCountry_Score"] = np.where(
    priority_df["Mean_JS"].notna(),
    1.0,
    0.5
)


# ---------------------------------------------------------
# 5. Composite priority score
# ---------------------------------------------------------

priority_df["Priority_Score"] = (
    0.35 * priority_df["Severity_Score"]
    + 0.25 * priority_df["Stability_Score"]
    + 0.25 * priority_df["Actionability_Score"]
    + 0.15 * priority_df["CrossCountry_Score"]
)

priority_df["Priority_Score_100"] = (
    priority_df["Priority_Score"] * 100
).round(1)


# ---------------------------------------------------------
# 6. Intervention priority level
# ---------------------------------------------------------

priority_df["Priority_Level"] = pd.cut(
    priority_df["Priority_Score_100"],
    bins=[-np.inf, 45, 65, np.inf],
    labels=[
        "Priority III — Monitoring",
        "Priority II — Targeted",
        "Priority I — High"
    ]
)


# ---------------------------------------------------------
# 7. Exclude outcome-only variables from intervention ranking
# ---------------------------------------------------------

priority_df["Eligible_for_Intervention"] = np.where(
    priority_df["Modifiability"] == "Outcome-related",
    "No",
    "Yes"
)


# ---------------------------------------------------------
# Sort
# ---------------------------------------------------------

priority_df = (
    priority_df
    .sort_values(
        "Priority_Score_100",
        ascending=False
    )
    .reset_index(drop=True)
)


print("Data-Driven Intervention Priority Ranking")
print("=" * 145)

priority_display = priority_df[
    [
        "Feature",
        "Factor_Domain",
        "Modifiability",
        "Fatal_Relevance",
        "Severity_Score",
        "Stability_Score",
        "Actionability_Score",
        "CrossCountry_Score",
        "Priority_Score_100",
        "Priority_Level",
        "Eligible_for_Intervention"
    ]
]

print(
    priority_display
    .round(3)
    .to_string(index=False)
)

In [ ]:
# Cell 8 — Revised intervention priority score
# Stability is based on relative variability (CV), not raw SHAP SD.

priority_final = evidence_master.copy()


# ==========================================================
# 1. Severity relevance
# ==========================================================

max_fatal = priority_final["Fatal_Relevance"].max()

priority_final["Severity_Score"] = (
    priority_final["Fatal_Relevance"] / max_fatal
).clip(0, 1)


# ==========================================================
# 2. Relative SHAP stability
# CV = SD / Mean Importance
# Choose available evidence from UK and/or Ethiopia
# ==========================================================

priority_final["UK_SHAP_CV"] = (
    priority_final["UK_SHAP_SD"]
    / priority_final["UK_Stable_SHAP_%"]
)

priority_final["Ethiopia_SHAP_CV"] = (
    priority_final["Ethiopia_SHAP_SD"]
    / priority_final["Ethiopia_Stable_SHAP_%"]
)

# If both countries exist, use their mean relative variability.
# If only one exists, use that country's CV.
priority_final["Mean_SHAP_CV"] = (
    priority_final[
        ["UK_SHAP_CV", "Ethiopia_SHAP_CV"]
    ]
    .mean(axis=1, skipna=True)
)

# Convert relative variability into stability.
# CV = 0 -> score 1
# Increasing CV -> progressively lower stability.
priority_final["Stability_Score"] = (
    1 / (1 + priority_final["Mean_SHAP_CV"])
)

priority_final["Stability_Score"] = (
    priority_final["Stability_Score"]
    .fillna(0.5)
    .clip(0, 1)
)


# ==========================================================
# 3. Actionability
# ==========================================================

actionability_scores = {
    "Directly modifiable": 1.00,
    "Directly/partly modifiable": 0.90,
    "Potentially modifiable": 0.85,
    "Partly modifiable": 0.75,
    "Indirectly modifiable": 0.60,
    "Not modifiable — targetable": 0.45,
    "Outcome-related": 0.00,
    "To be classified": 0.30
}

priority_final["Actionability_Score"] = (
    priority_final["Modifiability"]
    .map(actionability_scores)
    .fillna(0.30)
)


# ==========================================================
# 4. Cross-country evidence
# ==========================================================

priority_final["CrossCountry_Score"] = np.where(
    priority_final["Mean_JS"].notna(),
    1.0,
    0.5
)


# ==========================================================
# 5. Revised composite score
# ==========================================================

priority_final["Priority_Score"] = (
    0.35 * priority_final["Severity_Score"]
    + 0.25 * priority_final["Stability_Score"]
    + 0.25 * priority_final["Actionability_Score"]
    + 0.15 * priority_final["CrossCountry_Score"]
)

priority_final["Priority_Score_100"] = (
    100 * priority_final["Priority_Score"]
).round(1)


# ==========================================================
# 6. Priority categories
# ==========================================================

priority_final["Priority_Level"] = pd.cut(
    priority_final["Priority_Score_100"],
    bins=[-np.inf, 45, 65, np.inf],
    labels=[
        "Priority III — Monitoring",
        "Priority II — Targeted",
        "Priority I — High"
    ]
)


# ==========================================================
# 7. Intervention eligibility
# ==========================================================

priority_final["Eligible_for_Intervention"] = np.where(
    priority_final["Modifiability"] == "Outcome-related",
    "No",
    "Yes"
)


priority_final = (
    priority_final
    .sort_values(
        "Priority_Score_100",
        ascending=False
    )
    .reset_index(drop=True)
)


# ==========================================================
# Display
# ==========================================================

display_cols = [
    "Feature",
    "Factor_Domain",
    "Modifiability",
    "Fatal_Relevance",
    "Mean_SHAP_CV",
    "Stability_Score",
    "Actionability_Score",
    "CrossCountry_Score",
    "Priority_Score_100",
    "Priority_Level",
    "Eligible_for_Intervention"
]

print(
    "Revised Data-Driven Intervention Priority Ranking"
)

print("=" * 150)

print(
    priority_final[
        display_cols
    ]
    .round(3)
    .to_string(index=False)
)

In [ ]:
# Cell 9 — Save final intervention-priority ranking

output_file = TABLE_DIR / "Table_Final_Intervention_Priority_Ranking.csv"

priority_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Final intervention-priority ranking saved:")
print(output_file)

print("\nPriority distribution:")
print(priority_final["Priority_Level"].value_counts())

print("\nPriority I features:")
print(
    priority_final.loc[
        (priority_final["Priority_Level"] == "Priority I — High") &
        (priority_final["Eligible_for_Intervention"] == "Yes"),
        [
            "Feature",
            "Factor_Domain",
            "Priority_Score_100"
        ]
    ].to_string(index=False)
)

In [ ]:
# Cell 10 — Evidence-Based Intervention Matrix
#
# IMPORTANT:
# These interventions are linked to the XAI-derived intervention targets,
# but their effectiveness is supported by external road-safety evidence.
# SHAP importance itself is NOT interpreted as causal evidence.

intervention_data = [

    # =====================================================
    # 1. JUNCTION / INTERSECTION
    # =====================================================
    {
        "Feature": "junction_common",
        "Intervention_Target": "Intersection conflict reduction",
        "Recommended_Intervention":
            "Intersection redesign, appropriate roundabout conversion, "
            "dedicated turning movements, improved signal control, "
            "enhanced signing and pavement markings",
        "Mechanism":
            "Reduce conflict points, improve intersection recognition, "
            "and reduce high-severity crossing and turning conflicts",
        "Implementation_Level": "Infrastructure / Traffic engineering",
        "External_Evidence_Source": "FHWA Proven Safety Countermeasures",
        "Evidence_Strength": "High"
    },

    # =====================================================
    # 2. VEHICLE COUNT
    # =====================================================
    {
        "Feature": "vehicle_count",
        "Intervention_Target": "Multi-vehicle conflict management",
        "Recommended_Intervention":
            "Speed management, conflict-point reduction, access management, "
            "turning-lane control and intersection traffic management",
        "Mechanism":
            "Reduce collision energy and opportunities for multi-vehicle conflicts",
        "Implementation_Level": "Traffic management / Infrastructure",
        "External_Evidence_Source": "WHO Speed Management; FHWA",
        "Evidence_Strength": "High"
    },

    # =====================================================
    # 3. ROAD SURFACE
    # =====================================================
    {
        "Feature": "surface_common",
        "Intervention_Target": "Road-surface safety",
        "Recommended_Intervention":
            "Pavement friction monitoring, high-friction surface treatment "
            "at high-risk locations, drainage improvement and timely surface maintenance",
        "Mechanism":
            "Improve tire-road friction and vehicle control under adverse surface conditions",
        "Implementation_Level": "Infrastructure / Maintenance",
        "External_Evidence_Source": "FHWA Proven Safety Countermeasures",
        "Evidence_Strength": "High"
    },

    # =====================================================
    # 4. ACCIDENT AREA
    # =====================================================
    {
        "Feature": "Area_accident_occured",
        "Intervention_Target": "High-risk location management",
        "Recommended_Intervention":
            "Network screening, road safety audits and targeted engineering "
            "treatment of recurrent high-risk locations",
        "Mechanism":
            "Identify location-specific hazards and direct resources toward "
            "sites with elevated severe-crash risk",
        "Implementation_Level": "Network / Infrastructure planning",
        "External_Evidence_Source": "FHWA Road Safety Audit / Safe System",
        "Evidence_Strength": "High"
    },

    # =====================================================
    # 5. LIGHTING
    # =====================================================
    {
        "Feature": "light_common",
        "Intervention_Target": "Nighttime visibility",
        "Recommended_Intervention":
            "Adequate roadway and intersection lighting, targeted lighting "
            "at crossings and conflict points, and systematic lighting maintenance",
        "Mechanism":
            "Increase visibility and available reaction time for drivers "
            "and vulnerable road users",
        "Implementation_Level": "Infrastructure",
        "External_Evidence_Source": "FHWA Lighting Countermeasure",
        "Evidence_Strength": "High"
    },

    # =====================================================
    # 6. WEATHER
    # =====================================================
    {
        "Feature": "weather_common",
        "Intervention_Target": "Adverse-weather risk management",
        "Recommended_Intervention":
            "Dynamic speed management, warning systems, variable-message information "
            "and weather-responsive traffic management",
        "Mechanism":
            "Adapt traffic operation and driver behaviour to temporary environmental risk",
        "Implementation_Level": "Traffic operations / ITS",
        "External_Evidence_Source": "WHO Speed Management / Safe System",
        "Evidence_Strength": "Moderate-High"
    },

    # =====================================================
    # 7. MOTORCYCLE
    # =====================================================
    {
        "Feature": "has_motorcycle",
        "Intervention_Target": "Powered two-wheeler safety",
        "Recommended_Intervention":
            "Safe-speed policies, helmet-law enforcement, motorcycle-aware "
            "road design and targeted high-risk corridor management",
        "Mechanism":
            "Reduce crash probability and injury severity among vulnerable powered two-wheelers",
        "Implementation_Level": "Policy / Enforcement / Infrastructure",
        "External_Evidence_Source": "WHO Powered Two- and Three-Wheeler Safety",
        "Evidence_Strength": "High"
    },

    # =====================================================
    # 8. TWO-WHEELERS
    # =====================================================
    {
        "Feature": "has_two_wheeler",
        "Intervention_Target": "Two-wheeler conflict protection",
        "Recommended_Intervention":
            "Safer intersection design, visibility enhancement, speed reduction "
            "and separation or protection where contextually appropriate",
        "Mechanism":
            "Reduce exposure to high-energy conflicts involving vulnerable road users",
        "Implementation_Level": "Infrastructure / Speed management",
        "External_Evidence_Source": "WHO Safe System; FHWA",
        "Evidence_Strength": "High"
    }
]


intervention_matrix = pd.DataFrame(intervention_data)


# ---------------------------------------------------------
# Attach our XAI-derived priority scores
# ---------------------------------------------------------

intervention_matrix = intervention_matrix.merge(
    priority_final[
        [
            "Feature",
            "Priority_Score_100",
            "Priority_Level",
            "Fatal_Relevance"
        ]
    ],
    on="Feature",
    how="left"
)


# ---------------------------------------------------------
# Reorder by project-derived priority
# ---------------------------------------------------------

intervention_matrix = (
    intervention_matrix
    .sort_values(
        "Priority_Score_100",
        ascending=False
    )
    .reset_index(drop=True)
)


print("XAI-to-Intervention Matrix — Priority I")
print("=" * 150)

print(
    intervention_matrix[
        [
            "Feature",
            "Priority_Score_100",
            "Intervention_Target",
            "Implementation_Level",
            "Evidence_Strength"
        ]
    ].to_string(index=False)
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

output_file = (
    TABLE_DIR /
    "Table_XAI_to_Intervention_Matrix_Priority_I.csv"
)

intervention_matrix.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(output_file)

In [ ]:
# Cell 11 — Cross-Country Intervention Transferability Matrix

transferability_df = intervention_matrix.copy()


# ==========================================================
# Attach domain-shift evidence
# ==========================================================

transferability_df = transferability_df.merge(
    domain_master[
        [
            "Feature",
            "JS_UK_France",
            "JS_UK_Ethiopia",
            "JS_France_Ethiopia",
            "Mean_JS"
        ]
    ],
    on="Feature",
    how="left"
)


# ==========================================================
# Classify distribution shift
#
# These categories are operational categories defined for
# this framework, not universal JS-distance thresholds.
# ==========================================================

def classify_shift(js):

    if pd.isna(js):
        return "Country-specific / unavailable"

    elif js < 0.10:
        return "Low"

    elif js < 0.20:
        return "Moderate"

    else:
        return "High"


transferability_df["Domain_Shift_Level"] = (
    transferability_df["Mean_JS"]
    .apply(classify_shift)
)


# ==========================================================
# Translate domain shift into intervention-transfer guidance
# ==========================================================

def transfer_guidance(row):

    shift = row["Domain_Shift_Level"]

    if shift == "Low":
        return (
            "Higher transferability — standard intervention "
            "principles may be considered with routine local adaptation"
        )

    elif shift == "Moderate":
        return (
            "Conditional transferability — local calibration "
            "and contextual adaptation recommended"
        )

    elif shift == "High":
        return (
            "Limited direct transferability — local validation "
            "and country-specific intervention design required"
        )

    else:
        return (
            "Country-specific evidence — do not extrapolate "
            "without external/local validation"
        )


transferability_df["Transferability_Guidance"] = (
    transferability_df.apply(
        transfer_guidance,
        axis=1
    )
)


# ==========================================================
# Pair-specific transferability
# ==========================================================

for col, label in [
    ("JS_UK_France", "UK_France"),
    ("JS_UK_Ethiopia", "UK_Ethiopia"),
    ("JS_France_Ethiopia", "France_Ethiopia")
]:

    transferability_df[
        f"{label}_Shift_Level"
    ] = transferability_df[col].apply(classify_shift)


# ==========================================================
# Display
# ==========================================================

display_cols = [
    "Feature",
    "Priority_Score_100",
    "Mean_JS",
    "Domain_Shift_Level",
    "UK_France_Shift_Level",
    "UK_Ethiopia_Shift_Level",
    "France_Ethiopia_Shift_Level",
    "Transferability_Guidance"
]

print(
    "Cross-Country Intervention Transferability Matrix"
)

print("=" * 170)

print(
    transferability_df[
        display_cols
    ]
    .round(3)
    .to_string(index=False)
)


# ==========================================================
# Save
# ==========================================================

output_file = (
    TABLE_DIR /
    "Table_Intervention_Transferability_Matrix.csv"
)

transferability_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(output_file)

In [ ]:
# Cell 12 — Implementation Decision Matrix
#
# Converts XAI priority + modifiability + domain shift into
# implementation guidance.
#
# IMPORTANT:
# This framework prioritizes intervention targets.
# It does NOT estimate causal treatment effects.

implementation_df = transferability_df.copy()


# ==========================================================
# 1. Define implementation pathway
# ==========================================================

def implementation_pathway(row):

    shift = row["Domain_Shift_Level"]
    score = row["Priority_Score_100"]
    feature = row["Feature"]

    # Country-specific variable
    if shift == "Country-specific / unavailable":
        return "Local validation required"

    # High distribution shift
    if shift == "High":
        return "Local validation required"

    # Very high-priority + relatively transferable
    if score >= 70 and shift in ["Low", "Moderate"]:
        return "Immediate / System-level"

    # Remaining Priority-I interventions
    return "Adaptive / Targeted"


implementation_df["Implementation_Pathway"] = (
    implementation_df.apply(
        implementation_pathway,
        axis=1
    )
)


# ==========================================================
# 2. Define implementation interpretation
# ==========================================================

pathway_explanation = {

    "Immediate / System-level":
        "Strong intervention priority with comparatively acceptable "
        "cross-country transportability. The general safety principle "
        "can be prioritized, while operational details should still "
        "be locally adapted.",

    "Adaptive / Targeted":
        "Intervention is supported as a priority target, but implementation "
        "should be adapted to local exposure patterns, infrastructure, "
        "traffic composition, and risk context.",

    "Local validation required":
        "The factor is important but shows substantial domain shift or "
        "country-specific measurement. Local risk assessment should precede "
        "selection of the specific intervention design."
}

implementation_df["Implementation_Interpretation"] = (
    implementation_df["Implementation_Pathway"]
    .map(pathway_explanation)
)


# ==========================================================
# 3. Add decision rationale
# ==========================================================

def make_rationale(row):

    return (
        f"Priority={row['Priority_Score_100']:.1f}; "
        f"Mean JS="
        f"{row['Mean_JS']:.3f}"
        if pd.notna(row["Mean_JS"])
        else
        f"Priority={row['Priority_Score_100']:.1f}; "
        f"cross-country JS unavailable"
    )


implementation_df["Decision_Rationale"] = (
    implementation_df.apply(
        make_rationale,
        axis=1
    )
)


# ==========================================================
# 4. Create final implementation table
# ==========================================================

implementation_final = implementation_df[
    [
        "Feature",
        "Intervention_Target",
        "Recommended_Intervention",
        "Priority_Score_100",
        "Domain_Shift_Level",
        "Implementation_Pathway",
        "Implementation_Interpretation",
        "Decision_Rationale",
        "Evidence_Strength"
    ]
].copy()


# Sort by pathway and priority
pathway_order = {
    "Immediate / System-level": 1,
    "Adaptive / Targeted": 2,
    "Local validation required": 3
}

implementation_final["_order"] = (
    implementation_final["Implementation_Pathway"]
    .map(pathway_order)
)

implementation_final = (
    implementation_final
    .sort_values(
        ["_order", "Priority_Score_100"],
        ascending=[True, False]
    )
    .drop(columns="_order")
    .reset_index(drop=True)
)


# ==========================================================
# 5. Display concise result
# ==========================================================

print("Final XAI-Guided Implementation Decision Matrix")
print("=" * 150)

print(
    implementation_final[
        [
            "Feature",
            "Priority_Score_100",
            "Domain_Shift_Level",
            "Implementation_Pathway",
            "Intervention_Target"
        ]
    ].to_string(index=False)
)


# ==========================================================
# 6. Summary
# ==========================================================

print("\nImplementation pathway distribution:")
print(
    implementation_final[
        "Implementation_Pathway"
    ].value_counts()
)


# ==========================================================
# 7. Save
# ==========================================================

output_file = (
    TABLE_DIR /
    "Table_Final_XAI_Implementation_Decision_Matrix.csv"
)

implementation_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(output_file)

In [ ]:
# Cell 13 — Improved publication-ready methodological framework
# Larger fonts + colored blocks

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from pathlib import Path

fig, ax = plt.subplots(figsize=(18, 11))

ax.set_xlim(0, 15)
ax.set_ylim(0, 10.5)
ax.axis("off")


# ============================================================
# Color palette
# ============================================================

COLORS = {
    "data": "#D9EAF7",
    "harmonization": "#DDEFD8",
    "prediction": "#FFE4B5",
    "xai": "#E6D9F2",
    "robustness": "#F8D7DA",

    "severity": "#FCE8D5",
    "actionability": "#D9F0EE",
    "domain": "#E3E4FA",

    "priority": "#FFF2B2",

    "immediate": "#CDECCF",
    "adaptive": "#FFE0A8",
    "local": "#F4C7C3"
}


# ============================================================
# Helper functions
# ============================================================

def add_box(
    x,
    y,
    w,
    h,
    title,
    text,
    facecolor,
    title_size=14,
    text_size=11
):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.05,rounding_size=0.12",
        linewidth=1.8,
        edgecolor="#444444",
        facecolor=facecolor
    )

    ax.add_patch(box)

    ax.text(
        x + w / 2,
        y + h * 0.70,
        title,
        ha="center",
        va="center",
        fontsize=title_size,
        fontweight="bold"
    )

    ax.text(
        x + w / 2,
        y + h * 0.32,
        text,
        ha="center",
        va="center",
        fontsize=text_size,
        linespacing=1.25
    )


def arrow(x1, y1, x2, y2):
    ax.add_patch(
        FancyArrowPatch(
            (x1, y1),
            (x2, y2),
            arrowstyle="-|>",
            mutation_scale=20,
            linewidth=1.8,
            color="#555555"
        )
    )


# ============================================================
# TOP PIPELINE
# ============================================================

add_box(
    0.3, 7.6, 2.4, 1.6,
    "1. Multi-Country Data",
    "UK\nFrance\nEthiopia",
    COLORS["data"]
)

add_box(
    3.1, 7.6, 2.4, 1.6,
    "2. Harmonization",
    "Common predictors\nCommon severity classes",
    COLORS["harmonization"]
)

add_box(
    5.9, 7.6, 2.4, 1.6,
    "3. Prediction",
    "Cost-sensitive\nCatBoost",
    COLORS["prediction"]
)

add_box(
    8.7, 7.6, 2.4, 1.6,
    "4. Explainability",
    "Global SHAP\nClass-specific SHAP",
    COLORS["xai"]
)

add_box(
    11.5, 7.6, 3.0, 1.6,
    "5. Robustness",
    "Repeated validation\nSHAP stability",
    COLORS["robustness"]
)

arrow(2.7, 8.4, 3.1, 8.4)
arrow(5.5, 8.4, 5.9, 8.4)
arrow(8.3, 8.4, 8.7, 8.4)
arrow(11.1, 8.4, 11.5, 8.4)


# ============================================================
# SECOND LEVEL
# ============================================================

add_box(
    1.5, 4.8, 3.3, 1.7,
    "6. Severity Relevance",
    "Fatal-class relevance\n+\nfeature importance",
    COLORS["severity"]
)

add_box(
    5.85, 4.8, 3.3, 1.7,
    "7. Actionability",
    "Modifiable\nTargetable\nOutcome-related",
    COLORS["actionability"]
)

add_box(
    10.2, 4.8, 3.3, 1.7,
    "8. Domain Shift",
    "Jensen-Shannon distance\n+\nCross-country transfer",
    COLORS["domain"]
)

arrow(9.9, 7.6, 3.2, 6.5)
arrow(12.9, 7.6, 11.8, 6.5)


# ============================================================
# PRIORITY LAYER
# ============================================================

add_box(
    4.2, 2.45, 6.6, 1.55,
    "9. Intervention Priority Score",
    "Severity relevance  +  Stability  +  Actionability  +  Cross-country evidence",
    COLORS["priority"],
    title_size=15,
    text_size=11.5
)

arrow(3.15, 4.8, 5.8, 4.0)
arrow(7.5, 4.8, 7.5, 4.0)
arrow(11.85, 4.8, 9.2, 4.0)


# ============================================================
# FINAL DECISION LAYER
# ============================================================

add_box(
    0.6, 0.25, 4.0, 1.45,
    "Immediate / System-level",
    "General safety principle prioritized\nwith local operational adaptation",
    COLORS["immediate"],
    title_size=14,
    text_size=10.5
)

add_box(
    5.5, 0.25, 4.0, 1.45,
    "Adaptive / Targeted",
    "Targeted intervention with\ncontext-specific adaptation",
    COLORS["adaptive"],
    title_size=14,
    text_size=10.5
)

add_box(
    10.4, 0.25, 4.0, 1.45,
    "Local Validation Required",
    "High domain shift or\ncountry-specific evidence",
    COLORS["local"],
    title_size=14,
    text_size=10.5
)

arrow(7.5, 2.45, 2.6, 1.7)
arrow(7.5, 2.45, 7.5, 1.7)
arrow(7.5, 2.45, 12.4, 1.7)


# ============================================================
# MAIN TITLE
# ============================================================

ax.text(
    7.5,
    10.05,
    "XAI-Guided Cross-Country Road-Safety Intervention Framework",
    ha="center",
    va="center",
    fontsize=21,
    fontweight="bold"
)

ax.text(
    7.5,
    9.65,
    "From multicountry crash data to evidence-guided implementation decisions",
    ha="center",
    va="center",
    fontsize=13
)


# ============================================================
# SAVE
# ============================================================

plt.tight_layout()

figure_path = (
    FIGURE_DIR /
    "Figure_XAI_Intervention_Framework.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:")
print(figure_path)

In [ ]:
# Cell 14 — Sensitivity analysis of intervention-priority weights
#
# Tests whether the top-ranked intervention targets remain stable
# under several plausible alternative weighting schemes.

weight_scenarios = {
    "Primary":
        {
            "Severity": 0.35,
            "Stability": 0.25,
            "Actionability": 0.25,
            "CrossCountry": 0.15
        },

    "Severity_Emphasis":
        {
            "Severity": 0.45,
            "Stability": 0.20,
            "Actionability": 0.20,
            "CrossCountry": 0.15
        },

    "Actionability_Emphasis":
        {
            "Severity": 0.30,
            "Stability": 0.20,
            "Actionability": 0.35,
            "CrossCountry": 0.15
        },

    "Stability_Emphasis":
        {
            "Severity": 0.30,
            "Stability": 0.35,
            "Actionability": 0.20,
            "CrossCountry": 0.15
        },

    "CrossCountry_Emphasis":
        {
            "Severity": 0.30,
            "Stability": 0.20,
            "Actionability": 0.20,
            "CrossCountry": 0.30
        },

    "Equal_Weights":
        {
            "Severity": 0.25,
            "Stability": 0.25,
            "Actionability": 0.25,
            "CrossCountry": 0.25
        }
}


sensitivity_results = []

for scenario, weights in weight_scenarios.items():

    temp = priority_final.copy()

    temp["Sensitivity_Priority_Score"] = (
        weights["Severity"] *
        temp["Severity_Score"]
        +
        weights["Stability"] *
        temp["Stability_Score"]
        +
        weights["Actionability"] *
        temp["Actionability_Score"]
        +
        weights["CrossCountry"] *
        temp["CrossCountry_Score"]
    ) * 100

    temp["Sensitivity_Rank"] = (
        temp["Sensitivity_Priority_Score"]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    temp["Scenario"] = scenario

    sensitivity_results.append(
        temp[
            [
                "Scenario",
                "Feature",
                "Sensitivity_Priority_Score",
                "Sensitivity_Rank"
            ]
        ]
    )


priority_sensitivity_long = pd.concat(
    sensitivity_results,
    ignore_index=True
)


# ==========================================================
# Summarize rank stability
# ==========================================================

priority_sensitivity_summary = (
    priority_sensitivity_long
    .groupby("Feature")
    .agg(
        Mean_Rank=("Sensitivity_Rank", "mean"),
        SD_Rank=("Sensitivity_Rank", "std"),
        Best_Rank=("Sensitivity_Rank", "min"),
        Worst_Rank=("Sensitivity_Rank", "max"),
        Mean_Score=("Sensitivity_Priority_Score", "mean"),
        SD_Score=("Sensitivity_Priority_Score", "std")
    )
    .reset_index()
)

priority_sensitivity_summary = (
    priority_sensitivity_summary
    .sort_values(
        ["Mean_Rank", "Mean_Score"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


print(
    "Intervention Priority Sensitivity Analysis"
)

print("=" * 100)

print(
    priority_sensitivity_summary
    .round(2)
    .to_string(index=False)
)


# ==========================================================
# Top 8 features in each scenario
# ==========================================================

print("\n\nTop 8 features under each weighting scenario")
print("=" * 85)

for scenario in weight_scenarios.keys():

    temp = (
        priority_sensitivity_long[
            priority_sensitivity_long["Scenario"]
            == scenario
        ]
        .sort_values(
            "Sensitivity_Rank"
        )
        .head(8)
    )

    print(f"\n{scenario}")
    print("-" * 65)

    print(
        temp[
            [
                "Sensitivity_Rank",
                "Feature",
                "Sensitivity_Priority_Score"
            ]
        ]
        .round(2)
        .to_string(index=False)
    )

In [ ]:
# Cell 15 — Measure how consistently each feature appears
# among the top-eight intervention priorities.

top_n = 8

top_features_by_scenario = {}

for scenario in weight_scenarios.keys():

    temp = (
        priority_sensitivity_long[
            priority_sensitivity_long["Scenario"]
            == scenario
        ]
        .sort_values(
            "Sensitivity_Rank"
        )
        .head(top_n)
    )

    top_features_by_scenario[scenario] = set(
        temp["Feature"]
    )


feature_top_counts = []

for feature in priority_final["Feature"]:

    count = sum(
        feature in feature_set
        for feature_set in
        top_features_by_scenario.values()
    )

    feature_top_counts.append({
        "Feature": feature,
        "Top8_Count": count,
        "Top8_Percent":
            count / len(weight_scenarios) * 100
    })


top8_stability_df = pd.DataFrame(
    feature_top_counts
)

top8_stability_df = (
    top8_stability_df
    .sort_values(
        ["Top8_Count", "Feature"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print(
    "Top-8 Intervention Priority Stability"
)

print("=" * 70)

print(
    top8_stability_df
    .round(1)
    .to_string(index=False)
)


# Save results
priority_sensitivity_long.to_csv(
    TABLE_DIR /
    "Table_Intervention_Priority_Sensitivity_All_Scenarios.csv",
    index=False
)

priority_sensitivity_summary.to_csv(
    TABLE_DIR /
    "Table_Intervention_Priority_Sensitivity_Summary.csv",
    index=False
)

top8_stability_df.to_csv(
    TABLE_DIR /
    "Table_Intervention_Top8_Stability.csv",
    index=False
)

print("\nSensitivity-analysis tables saved.")

In [ ]:
# Cell 16 — Visualize intervention-priority robustness across weighting scenarios.

import matplotlib.pyplot as plt

plot_df = (
    top8_stability_df[
        top8_stability_df["Top8_Count"] > 0
    ]
    .sort_values(
        ["Top8_Percent", "Feature"],
        ascending=[True, True]
    )
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

bars = ax.barh(
    plot_df["Feature"],
    plot_df["Top8_Percent"]
)

ax.set_xlabel(
    "Presence in Top-8 priority set (%)",
    fontsize=12
)

ax.set_ylabel(
    "Intervention target",
    fontsize=12
)

ax.set_title(
    "Robustness of Intervention Priorities Across Weighting Scenarios",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlim(0, 110)

for bar, value in zip(
    bars,
    plot_df["Top8_Percent"]
):

    ax.text(
        value + 1.5,
        bar.get_y() + bar.get_height()/2,
        f"{value:.1f}%",
        va="center",
        fontsize=10
    )

plt.tight_layout()

figure_path = (
    FIGURE_DIR /
    "Figure_Intervention_Priority_Sensitivity.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:")
print(figure_path)

In [ ]:
# Cell 17 — Create the final robust intervention-priority shortlist.

robust_interventions = (
    priority_final[
        priority_final["Feature"].isin(
            top8_stability_df.loc[
                top8_stability_df["Top8_Percent"] >= 80,
                "Feature"
            ]
        )
    ]
    .merge(
        top8_stability_df[
            ["Feature", "Top8_Count", "Top8_Percent"]
        ],
        on="Feature",
        how="left"
    )
)

robust_interventions = robust_interventions[
    [
        "Feature",
        "Factor_Domain",
        "Priority_Score_100",
        "Priority_Level",
        "Fatal_Relevance",
        "Modifiability",
        "Top8_Count",
        "Top8_Percent"
    ]
].sort_values(
    "Priority_Score_100",
    ascending=False
)

print(
    "Final Robust Intervention Priorities"
)

print("=" * 110)

print(
    robust_interventions
    .round(2)
    .to_string(index=False)
)

robust_interventions.to_csv(
    TABLE_DIR /
    "Table_Final_Robust_Intervention_Priorities.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nSaved: "
    "Table_Final_Robust_Intervention_Priorities.csv"
)

In [ ]:
# Cell 18 — Final Evidence-to-Action Matrix
#
# Integrates:
# 1) XAI evidence
# 2) Fatal-severity relevance
# 3) intervention priority
# 4) sensitivity robustness
# 5) cross-country transferability
# 6) implementation pathway

# ----------------------------------------------------------
# 1. Start from robust intervention shortlist
# ----------------------------------------------------------

final_action_matrix = robust_interventions.copy()


# ----------------------------------------------------------
# 2. Add intervention information
# ----------------------------------------------------------

intervention_cols = [
    "Feature",
    "Intervention_Target",
    "Recommended_Intervention",
    "Implementation_Level",
    "Evidence_Strength"
]

available_intervention_cols = [
    col for col in intervention_cols
    if col in intervention_matrix.columns
]

final_action_matrix = final_action_matrix.merge(
    intervention_matrix[available_intervention_cols],
    on="Feature",
    how="left"
)


# ----------------------------------------------------------
# 3. Add transferability evidence
# ----------------------------------------------------------

transfer_cols = [
    "Feature",
    "Mean_JS",
    "Domain_Shift_Level",
    "Transferability_Guidance"
]

final_action_matrix = final_action_matrix.merge(
    transferability_df[transfer_cols],
    on="Feature",
    how="left"
)


# ----------------------------------------------------------
# 4. Add final implementation pathway
# ----------------------------------------------------------

implementation_cols = [
    "Feature",
    "Implementation_Pathway"
]

final_action_matrix = final_action_matrix.merge(
    implementation_final[implementation_cols],
    on="Feature",
    how="left"
)


# ----------------------------------------------------------
# 5. Create concise evidence interpretation
# ----------------------------------------------------------

def evidence_summary(row):

    robustness = row["Top8_Percent"]

    if robustness == 100:
        robust_text = "Highly robust"
    elif robustness >= 80:
        robust_text = "Robust"
    else:
        robust_text = "Moderately robust"

    if pd.isna(row["Mean_JS"]):
        transfer_text = "Country-specific"
    elif row["Mean_JS"] < 0.10:
        transfer_text = "Low shift"
    elif row["Mean_JS"] < 0.20:
        transfer_text = "Moderate shift"
    else:
        transfer_text = "High shift"

    return (
        f"{robust_text}; "
        f"{transfer_text}"
    )


final_action_matrix["Evidence_Summary"] = (
    final_action_matrix.apply(
        evidence_summary,
        axis=1
    )
)


# ----------------------------------------------------------
# 6. Select final publication-oriented columns
# ----------------------------------------------------------

final_action_matrix = final_action_matrix[
    [
        "Feature",
        "Factor_Domain",
        "Fatal_Relevance",
        "Priority_Score_100",
        "Top8_Percent",
        "Evidence_Summary",
        "Intervention_Target",
        "Recommended_Intervention",
        "Domain_Shift_Level",
        "Implementation_Pathway",
        "Evidence_Strength"
    ]
].copy()


final_action_matrix = (
    final_action_matrix
    .sort_values(
        "Priority_Score_100",
        ascending=False
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# 7. Display
# ----------------------------------------------------------

print(
    "Final Evidence-to-Action Matrix — Robust Intervention Targets"
)

print("=" * 170)

print(
    final_action_matrix
    .round(2)
    .to_string(index=False)
)


# ----------------------------------------------------------
# 8. Save
# ----------------------------------------------------------

output_file = (
    TABLE_DIR /
    "Table_Final_Evidence_to_Action_Matrix.csv"
)

final_action_matrix.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(output_file)

In [ ]:
# Cell 19 — Final intervention framework summary

framework_summary = (
    final_action_matrix
    .groupby(
        "Implementation_Pathway",
        dropna=False
    )
    .agg(
        Number_of_Targets=("Feature", "count"),
        Mean_Priority_Score=("Priority_Score_100", "mean"),
        Mean_Robustness=("Top8_Percent", "mean")
    )
    .reset_index()
)

framework_summary[
    "Mean_Priority_Score"
] = framework_summary[
    "Mean_Priority_Score"
].round(1)

framework_summary[
    "Mean_Robustness"
] = framework_summary[
    "Mean_Robustness"
].round(1)


print("Final Intervention Framework Summary")
print("=" * 90)

print(
    framework_summary.to_string(
        index=False
    )
)


# Save
framework_summary.to_csv(
    TABLE_DIR /
    "Table_Final_Intervention_Framework_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nSaved: "
    "Table_Final_Intervention_Framework_Summary.csv"
)

In [ ]:
from pathlib import Path

print("=" * 100)
print("FINAL PROJECT OUTPUT INVENTORY")
print("=" * 100)

print("\nTABLES")
print("-" * 100)

table_files = sorted(TABLE_DIR.glob("*"))

for i, file in enumerate(table_files, 1):
    print(f"{i:02d}. {file.name}")


print("\n\nFIGURES")
print("-" * 100)

figure_files = sorted(FIGURE_DIR.glob("*"))

for i, file in enumerate(figure_files, 1):
    print(f"{i:02d}. {file.name}")


print("\n\nSUMMARY")
print("-" * 100)
print(f"Total tables/files in TABLE_DIR: {len(table_files)}")
print(f"Total figures in FIGURE_DIR:     {len(figure_files)}")